In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. PREPARAÇÃO DOS DADOS (Simulação Calibrada: N=52.160 | Faltas=4.327)
# ==============================================================================
total_n = 52160
faltas_n = 4327

# Categorias Prováveis de Motivos baseadas em Projetos de Telemedicina (TeleSAP)
motivos_lista = [
    "Problemas Técnicos/Internet", "Esquecimento", "Trabalho/Conflito de Horário",
    "Dificuldade com a Plataforma", "Melhora dos Sintomas", "Sem Justificativa (No-show)",
    "Falta de Documentação", "Erro de Agendamento/Sistema"
]

# Pesos para simular a realidade (Alguns motivos são mais frequentes)
pesos = [0.25, 0.20, 0.15, 0.15, 0.10, 0.08, 0.04, 0.03]

df_faltas = pd.DataFrame({
    'Motivo': np.random.choice(motivos_lista, size=faltas_n, p=pesos),
    'Dia_Semana': np.random.choice(['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta'], size=faltas_n)
})

# ==============================================================================
# 2. ANÁLISE DE PARETO (OS "VILÕES" DO ABSENTEÍSMO)
# ==============================================================================
pareto_data = df_faltas['Motivo'].value_counts().reset_index()
pareto_data.columns = ['Motivo', 'Frequencia']
pareto_data = pareto_data.sort_values(by='Frequencia', ascending=False)
pareto_data['Cum_Percentage'] = pareto_data['Frequencia'].cumsum() / pareto_data['Frequencia'].sum() * 100

# ==============================================================================
# 3. VISUALIZAÇÃO DOS MOTIVOS (GRÁFICO DE PARETO)
# ==============================================================================
fig, ax1 = plt.subplots(figsize=(14, 8))
sns.set_theme(style="white")

# Barras de Frequência
sns.barplot(x='Frequencia', y='Motivo', data=pareto_data, palette='magma', ax=ax1)
ax1.set_title(f"Análise de Causas Raiz do Absenteísmo (N={faltas_n} faltas)", fontsize=16, fontweight='bold')
ax1.set_xlabel("Quantidade de Ocorrências", fontsize=12)

# Linha de Pareto (Acumulado)
ax2 = ax1.twiny()
ax2.plot(pareto_data['Cum_Percentage'], pareto_data['Motivo'], color="#c0392b", marker="D", ms=7, label="Percentual Acumulado")
ax2.axvline(80, color="#c0392b", linestyle="--", alpha=0.5) # Linha dos 80%
ax2.set_xlabel("Percentual Acumulado (%)", fontsize=12)
ax2.set_xlim(0, 110)

plt.tight_layout()
plt.show()

# ==============================================================================
# 4. MAPA DE CALOR: MOTIVO vs DIA DA SEMANA
# ==============================================================================
plt.figure(figsize=(12, 6))
heatmap_data = pd.crosstab(df_faltas['Motivo'], df_faltas['Dia_Semana'])

# Reordenar dias para a ordem cronológica lógica
dias_ordem = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta']
heatmap_data = heatmap_data[dias_ordem]

sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='YlGnBu', cbar_kws={'label': 'Nº de Faltas'})
plt.title("Onde a Bimodalidade acontece: Motivos por Dia da Semana", fontsize=14, fontweight='bold')
plt.ylabel("")
plt.xlabel("")

plt.tight_layout()
plt.show()

# ==============================================================================
# 5. EXPORTAÇÃO SILENCIOSA DAS TABELAS
# ==============================================================================
# --- Exportando a Tabela do Pareto ---
df_pareto_export = pareto_data.copy()
df_pareto_export['Cum_Percentage'] = df_pareto_export['Cum_Percentage'].round(1)
df_pareto_export.rename(columns={
    'Frequencia': 'Quantidade de Faltas', 
    'Cum_Percentage': 'Percentual Acumulado (%)'
}, inplace=True)

arq_pareto = 'Tabela_Pareto_Motivos_Absenteismo.csv'
df_pareto_export.to_csv(arq_pareto, index=False, encoding='utf-8-sig')

# --- Exportando a Tabela do Mapa de Calor ---
# O reset_index joga os "Motivos" (que eram o índice) para uma coluna normal no Excel
df_heatmap_export = heatmap_data.reset_index()

arq_heatmap = 'Tabela_Heatmap_Faltas_Dias.csv'
df_heatmap_export.to_csv(arq_heatmap, index=False, encoding='utf-8-sig')

# ==============================================================================
# 6. DIAGNÓSTICO PARA GESTÃO
# ==============================================================================
top_motivo = pareto_data.iloc[0]['Motivo']
print("\n" + "="*80)
print(f"DIAGNÓSTICO EXECUTIVO DO ABSENTEÍSMO")
print("="*80)
print(f"1. O principal detrator da agenda é: {top_motivo}")
print(f"2. Os top 3 motivos explicam {pareto_data.iloc[2]['Cum_Percentage']:.1f}% de todas as faltas.")
print(f"3. Sugestão: Focar em 'Busca Ativa' para os motivos de {pareto_data.iloc[0]['Motivo']}.")
print("-" * 80)
print(f"📂 Tabelas exportadas com sucesso:")
print(f"  -> '{arq_pareto}'")
print(f"  -> '{arq_heatmap}'")
print("="*80 + "\n")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. CONFIGURAÇÃO DOS DADOS (Simulação Baseada nos Seus Números Reais)
# ==============================================================================
total_n = 52160
faltas_n = 4327
atendidos_n = total_n - faltas_n

datas = pd.date_range(start='2025-01-01', periods=total_n, freq='10min')

df = pd.DataFrame({
    'Data': datas,
    'Status': ['Atendido'] * atendidos_n + ['Paciente não compareceu'] * faltas_n
})

df = df.sample(frac=1).reset_index(drop=True)

# ==============================================================================
# 2. PROCESSAMENTO TEMPORAL
# ==============================================================================
df['Dia_Semana_Num'] = df['Data'].dt.dayofweek
df['Mes_Ano'] = df['Data'].dt.to_period('M')
dias_map = {0: 'Segunda', 1: 'Terça', 2: 'Quarta', 3: 'Quinta', 4: 'Sexta', 5: 'Sábado', 6: 'Domingo'}
df['Dia_Semana'] = df['Dia_Semana_Num'].map(dias_map)

# Cálculo agrupado por Dia da Semana (Para o gráfico de linha e labels)
df_semana = df.groupby(['Dia_Semana_Num', 'Dia_Semana']).agg(
    Total=('Status', 'count'),
    Faltas=('Status', lambda x: (x == 'Paciente não compareceu').sum())
).reset_index()
df_semana['Taxa_Abs'] = (df_semana['Faltas'] / df_semana['Total']) * 100

# Cálculo DIÁRIO (Para o Violin Plot mostrar a variação real de volume por dia)
df_diario = df.groupby([df['Data'].dt.date, 'Dia_Semana']).agg(
    Total=('Status', 'count'),
    Faltas=('Status', lambda x: (x == 'Paciente não compareceu').sum())
).reset_index()

# ==============================================================================
# 3. DASHBOARD VISUAL
# ==============================================================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- GRÁFICO 1: SAZONALIDADE MENSAL (Mantido em % para tendência) ---
df_mes = df.groupby('Mes_Ano').agg(
    Total=('Status', 'count'),
    Faltas=('Status', lambda x: (x == 'Paciente não compareceu').sum())
).reset_index()
df_mes['Taxa_Abs'] = (df_mes['Faltas'] / df_mes['Total']) * 100
df_mes['Mes_Nome'] = df_mes['Mes_Ano'].astype(str)

sns.lineplot(data=df_mes, x='Mes_Nome', y='Taxa_Abs', marker='o', linewidth=2.5, ax=axes[0], color='#2c3e50')
axes[0].fill_between(df_mes['Mes_Nome'], df_mes['Taxa_Abs'], alpha=0.1, color='#2c3e50')
axes[0].set_title("Sazonalidade Mensal do Absenteísmo", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Taxa de Falta (%)")
axes[0].tick_params(axis='x', rotation=45)

# --- GRÁFICO 2: VIOLIN PLOT (ABSENTEÍSMO EM NÚMEROS ABSOLUTOS) ---
ordem_dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']

# Mudança principal: y='Faltas' em vez de 'Taxa_Abs'
sns.violinplot(data=df_diario, x='Dia_Semana', y='Faltas', order=ordem_dias,
               palette="viridis", inner="quart", ax=axes[1])
sns.stripplot(data=df_diario, x='Dia_Semana', y='Faltas', order=ordem_dias,
              color='black', alpha=0.3, size=3, jitter=True, ax=axes[1])

axes[1].set_title(f"Volume Absoluto de Faltas por Dia (Total: {faltas_n})", fontsize=14, fontweight='bold')
axes[1].set_ylabel("Nº de Pacientes (Absoluto)")
axes[1].set_xlabel("")

# Ajuste dinâmico do limite inferior para os textos não sobreporem o gráfico
offset = df_diario['Faltas'].max() * 0.15

for i, dia in enumerate(ordem_dias):
    row = df_semana[df_semana['Dia_Semana'] == dia].iloc[0]
    axes[1].text(i, -offset, f"Total Faltas:\n{int(row['Faltas'])}",
                 ha='center', fontsize=9, fontweight='bold', color='#c0392b')

plt.tight_layout()
plt.show()

# ==============================================================================
# 4. EXPORTAÇÃO SILENCIOSA DAS TABELAS
# ==============================================================================
# --- Tabela 1: Sazonalidade Mensal ---
df_tabela_mes = df_mes[['Mes_Nome', 'Total', 'Faltas', 'Taxa_Abs']].copy()
df_tabela_mes.rename(columns={
    'Mes_Nome': 'Mês',
    'Total': 'Total de Agendamentos',
    'Faltas': 'Quantidade de Faltas',
    'Taxa_Abs': 'Taxa de Absenteísmo (%)'
}, inplace=True)
df_tabela_mes['Taxa de Absenteísmo (%)'] = df_tabela_mes['Taxa de Absenteísmo (%)'].round(2)

arq_mes = 'Tabela_Sazonalidade_Mensal.csv'
df_tabela_mes.to_csv(arq_mes, index=False, encoding='utf-8-sig')

# --- Tabela 2: Absenteísmo por Dia da Semana ---
df_tabela_semana = df_semana[['Dia_Semana', 'Total', 'Faltas', 'Taxa_Abs']].copy()
df_tabela_semana.rename(columns={
    'Dia_Semana': 'Dia da Semana',
    'Total': 'Total de Agendamentos',
    'Faltas': 'Quantidade de Faltas',
    'Taxa_Abs': 'Taxa de Absenteísmo (%)'
}, inplace=True)
df_tabela_semana['Taxa de Absenteísmo (%)'] = df_tabela_semana['Taxa de Absenteísmo (%)'].round(2)

# Garante a ordenação cronológica correta (Segunda a Domingo)
df_tabela_semana['Dia_Semana_Num'] = df_semana['Dia_Semana_Num']
df_tabela_semana = df_tabela_semana.sort_values('Dia_Semana_Num').drop(columns=['Dia_Semana_Num']).reset_index(drop=True)

arq_semana = 'Tabela_Absenteismo_Dia_Semana.csv'
df_tabela_semana.to_csv(arq_semana, index=False, encoding='utf-8-sig')

# ==============================================================================
# 5. RELATÓRIO EXECUTIVO
# ==============================================================================
print(f"\n📊 RELATÓRIO DE IMPACTO OPERACIONAL")
print("="*60)
for _, row in df_semana.sort_values('Dia_Semana_Num').iterrows():
    print(f" -> {row['Dia_Semana']:<10} | Faltas: {int(row['Faltas']):>4} pacientes | Média Diária: {int(row['Faltas']/52):>2}")

print("-" * 60)
print(f"📂 Tabelas exportadas silenciosamente com sucesso:")
print(f"  -> '{arq_mes}'")
print(f"  -> '{arq_semana}'")
print("="*60 + "\n")